# gem

> Simple utilities for working with Google's Gemini API

This notebook provides a minimal interface to Google's Gemini API. The goal is to make it dead simple to:

1. Generate text with just a prompt
2. Analyze files (PDFs, images, **MP4 videos**) 
3. Process videos (YouTube URLs or **local MP4 files**)

All through a single `gem()` function that just works.

In [1]:
#| default_exp gem

In [2]:
#| hide
from nbdev.showdoc import *

/Users/hamel/git/prompts/hamel/.venv/lib/python3.10/site-packages/fastprogress/fastprogress.py:107: UserWarning: Couldn't import ipywidgets properly, progress bar will use console behavior
  warn("Couldn't import ipywidgets properly, progress bar will use console behavior")


## Setup

First, make sure you have your Gemini API key set:

In [3]:
#| export
import os, time, mimetypes
from pathlib import Path
from fastcore.all import *
from google import genai
from google.genai import types
from functools import partial
from fastprogress import progress_bar

In [4]:
# export GEMINI_API_KEY='your-api-key'
assert os.environ.get("GEMINI_API_KEY"), "Please set GEMINI_API_KEY environment variable"

## Building blocks

Let's start with the simple helper functions that make everything work.

### Client creation

We need a Gemini client to talk to the API:

In [5]:
#|export
def _client():
    "Get Gemini client context manager"
    return genai.Client()

In [6]:
#|hide
c = _client()
assert c is not None
assert hasattr(c, 'models')

## Video upload

In [7]:
#|export
def upload_file(pth):
    if not Path(pth).exists(): raise ValueError(f"File {pth} does not exist.")
    with _client() as c:
        f = c.files.upload(file=pth)
        time.sleep(2)
        for i in progress_bar(range(30)):
            try:
                f = c.files.get(name=f.name)
                if f.state == 'ACTIVE': return f
                elif f.state == 'FAILED': raise Exception(f'File processing for {pth} failed.')
                time.sleep(10)
            except: pass # because the gemini file thing is jank
        raise Exception(f'Timeout processing {pth}')

In [8]:
myfile = upload_file("_videos/test_video.mp4")
assert myfile.state == 'ACTIVE'

 |----------------------------------------| 0.00% [0/30 00:00<?]

### Converting attachments to Parts

Gemini expects different types of content (files, URLs) to be wrapped in "Parts". This helper handles that conversion:

In [12]:
#| export
def _is_url(s):
    "Check if string is a URL"
    if not isinstance(s, str): return False
    return (s.startswith('http://') or 
            s.startswith('https://') or 
            s.startswith('www.') or 
            'youtube.com' in s or 
            'youtu.be' in s)

def _make_part(o):
    "Convert object to Gemini Part"
    if isinstance(o, types.File):
        return types.Part.from_uri(file_uri=o.uri, mime_type=o.mime_type)
    if isinstance(o, (str, Path)):
        p = Path(o)
        if p.exists():
            # Upload video and audio files via Files API (recommended for larger files)
            video_exts = {'.mp4', '.mpeg', '.mov', '.avi', '.flv', '.mpg', '.webm', '.wmv', '.3gpp'}
            audio_exts = {'.wav', '.mp3', '.aiff', '.aac', '.ogg', '.flac', '.m4a'}
            if p.suffix.lower() in video_exts | audio_exts:
                f = upload_file(o)
                return types.Part.from_uri(file_uri=f.uri, mime_type=f.mime_type)
            mime_map = {'.pdf': 'application/pdf', 
                        '.png': 'image/png', 
                        '.jpg': 'image/jpeg', 
                        '.jpeg': 'image/jpeg', 
                        '.gif': 'image/gif',
                        '.txt': 'text/plain',
                        '.vtt': 'text/plain',  # VTT treated as plain text
                        '.md': 'text/markdown',
                        '.json': 'text/plain',  # Gemini doesn't accept application/json
                        '.yaml': 'text/yaml',
                        '.yml': 'text/yaml',
                        '.toml': 'text/plain',  # Gemini doesn't accept application/toml
                        '.ipynb': 'text/plain'}  # Gemini doesn't accept application/x-ipynb+json
            mime = mime_map.get(p.suffix.lower())
            if mime is None:
                guessed_mime, _ = mimetypes.guess_type(str(p))
                if guessed_mime is None:
                    raise ValueError(f"Cannot determine MIME type for file: {p}. Unsupported extension: {p.suffix}")
                mime = guessed_mime
            return types.Part.from_bytes(mime_type=mime, data=p.read_bytes())
        elif _is_url(o): return types.Part.from_uri(file_uri=o, mime_type='video/*')
        else: raise ValueError(f"Could not parse file or url: {o}")
    return None

In [10]:
_part = _make_part('_videos/test_video.mp4')
_part

 |----------------------------------------| 0.00% [0/30 00:00<?]

Part(
  file_data=FileData(
    file_uri='https://generativelanguage.googleapis.com/v1beta/files/y328hzqczeyj',
    mime_type='video/mp4'
  )
)

In [14]:
#|hide
# Test text file formats
txt_part = _make_part('_test_files/sample.txt')
assert txt_part.inline_data.mime_type == 'text/plain'

vtt_part = _make_part('_test_files/sample.vtt')
assert vtt_part.inline_data.mime_type == 'text/plain'

md_part = _make_part('_test_files/sample.md')
assert md_part.inline_data.mime_type == 'text/markdown'

## The main interface

Now we can build our main `gem()` function that handles all use cases:

In [15]:
#| export
def gem(prompt, # Text prompt
        o=None, # Optional file/URL attachment or list of attachments
        model='gemini-2.5-flash',
        thinking=-1,
        search=False):
    "Generate content with Gemini"
    parts = [types.Part.from_text(text=prompt)]
    # Handle single attachment or list of attachments
    attachments = o if isinstance(o, list) else [o] if o else []
    for attachment in attachments:
        if part := _make_part(attachment): parts.insert(0, part)
    
    contents = types.Content(role='user', parts=parts) if attachments else prompt    
    config_dict = {
        'thinking_config': types.ThinkingConfig(thinking_budget=thinking),
        'response_mime_type': 'text/plain'
    }
    # Adjust media_resolution for videos for more tokens
    if any(p.file_data and p.file_data.mime_type.startswith('video') for p in parts):
        config_dict['media_resolution'] = 'MEDIA_RESOLUTION_LOW'
    config_dict['tools'] = []
    if search: config_dict['tools'].append(types.Tool(google_search=types.GoogleSearch()))
    cfg = types.GenerateContentConfig(**config_dict)
    with _client() as client:
        resp = client.models.generate_content(model=model, contents=contents, config=cfg)
    return resp.text

## Examples

One function handles everything:
- Just text? Pass a prompt.
- Have a file? Pass it as the second argument.
- Got a YouTube URL? Same thing.

Let's test it out:

## Text generation

The simplest case - just generate some text:

In [16]:
gem("Write a haiku about Python programming")

'Clear and simple lines,\nIndentation guides the way,\nCode starts to run free.'

## Video analysis

Perfect for creating YouTube chapters or summaries:

In [17]:
prompt = "5 word summary of this video."
gem(prompt, "https://youtu.be/1x3k0V2IITo")

'The speaker discusses the limitations of single vector search and introduces **late interaction models** (multi-vector models) as a solution. These models are shown to improve **out-of-domain generalization**, **long-context handling**, and **reasoning-intensive retrieval**. He also introduces **PyLate**, a library for training and evaluating late interaction models, which integrates well with the Hugging Face ecosystem.'

### Local MP4 Video Analysis

You can also analyze local MP4 video files:

In [18]:
# Example with local MP4 file (if you have one)
gem("Summarize this video in 3 sentences.", "_videos/test_video.mp4")

 |----------------------------------------| 0.00% [0/30 00:00<?]

'The video features a man conducting a brief test recording. During the recording, he recites the numbers "1 2 3, 4 5 6" and introduces himself as Hamil Hussain. He is bald, wears glasses and a dark polka-dotted shirt, and is seated indoors in front of a window.'

### File analysis

Great for extracting information from PDFs or images:

In [19]:
gem("3 sentence summary of this presentation.", "NewFrontiersInIR.pdf")

'This presentation introduces "New Frontiers in IR," focusing on enabling Information Retrieval systems to perform instruction following and reasoning, much like Large Language Models (LLMs). It proposes two main systems: "Promptriever," a fast, instruction-trained bi-encoder, and "Rank1," a powerful but slower reasoning-based reranker that leverages LLMs for test-time computation. Through these approaches, the research demonstrates that retrievers can be made promptable and capable of sophisticated reasoning, significantly improving performance on complex queries and unlocking new possibilities for search beyond traditional keyword matching.'

In [20]:
gem("What's in this image?", "anton.png")

'This image is a striking visual, likely a thumbnail for a video or article, set against a dark, deep blue or black background. It combines text, an emoji, a human face, and a technical diagram.\n\nHere\'s a breakdown of its contents:\n\n1.  **Text:**\n    *   In the upper left, large white text reads "Single Vector?".\n    *   Below that, a prominent stack of large, yellow, capitalized text spells out:\n        *   "YOU\'RE"\n        *   "MISSING"\n        *   "OUT"\n    *   On the right side, within a blue rectangular box, white text reads "RAG".\n\n2.  **Emoji:**\n    *   A yellow, sad or worried emoji with downturned eyes and mouth is positioned above and slightly to the left of the "YOU\'RE" text.\n\n3.  **Person:**\n    *   A young man with light skin, short brown hair, and a wide, friendly smile is prominently featured on the bottom left side of the image. He is wearing a white v-neck or t-shirt. He appears to be looking directly at the viewer.\n\n4.  **Diagram/Graphics:**\n    

### Text file analysis

Works with common text formats like .txt, .vtt, .md:

In [21]:
gem("What type of file is this?", "_test_files/sample.txt")

'This is a **plain text file**.\n\nSpecifically, the content itself mentions "Testing text/plain MIME type support," which confirms its type.'

In [22]:
gem("How many subtitle entries are in this file?", "_test_files/sample.vtt")

'There are **2** subtitle entries in this file.'

In [23]:
gem("List the items in this markdown file.", "_test_files/sample.md")

'The list items in this Markdown file are:\n\n*   Item 1\n*   Item 2'

### Change Model

You can also control the model and thinking time:

In [24]:
gem("What is Hamel Husain's current job?", model="gemini-2.5-pro")

"As of my last update, Hamel Husain's current job is **Head of Machine Learning at Outerbounds**.\n\nHe joined Outerbounds in November 2023. Outerbounds is a company focused on building tools for MLOps and is the commercial entity behind the popular open-source project **Metaflow**, which originated at Netflix.\n\nBefore this role, he was a well-known Staff Machine Learning Engineer at **Airtable**. He is also highly regarded in the data science community for his open-source work, including creating projects like `nbdev` and `fastpages`, and for his affiliation with **fast.ai**."

### Grounded Search

As you can see, grounded search is required to get things right sometimes!

In [25]:
gem("What is Hamel Husain's current job?.", search=True)

'Hamel Husain is currently working as an independent AI consultant, assisting companies in building AI products and operationalizing Large Language Models (LLMs). He also co-teaches a course titled "AI Evals for Engineers & PMs".\n\nPreviously, Hamel Husain held the position of Staff Machine Learning Engineer at GitHub. He has over 20 years of experience as a machine learning engineer, having worked with companies such as Airbnb and GitHub, and has contributed to numerous open-source machine learning tools.'

### Multiple Attachments

You can analyze multiple files/URLs at once by passing a list:

In [26]:
prompt = "Is this PDF and YouTube video related or are they different talks? Answer with very short yes/no answer."
gem(prompt, ["https://youtu.be/Trps2swgeOg?si=yK7CO0Zk4E1rfp6s", "NewFrontiersInIR.pdf"])

'No.'

In [27]:
gem(prompt, ["https://youtu.be/YB3b-wPbSH8?si=WI0LqflY5SYIsRz9", "NewFrontiersInIR.pdf"])

'Yes.'

In [28]:
gem("What do these slides and this video have in common in terms of content/subject matter if at all? Provide a 1 sentence summary of each.", ["NewFrontiersInIR.pdf", "_videos/test_video.mp4"])

 |█---------------------------------------| 3.33% [1/30 00:10<04:53]

'The video and the slides **do not share any common content or subject matter**.\n\n*   **Video Summary:** The video is a brief personal test recording where the speaker introduces himself and counts numbers to check the audio/video quality.\n*   **Slides Summary:** The slides present research on "New Frontiers in IR: Instruction Following and Reasoning," detailing how information retrieval systems can be designed to understand and execute natural language instructions and perform reasoning, similar to large language models, through models like Promptriever and Rank1.'

## Export -

In [29]:
#| hide
import nbdev; nbdev.nbdev_export()